# ERC-SVD: Complete Post-Training Compression Pipeline

This notebook presents the complete ERC-SVD workflow rather than a runnable experiment. Each section specifies the purpose, core operation, and output of one compression stage.

**Pipeline**

Model and calibration data $\rightarrow$ activation scaling matrices $\rightarrow$ partial-layer error analysis $\rightarrow$ rank allocation $\rightarrow$ residual-compensated SVD $\rightarrow$ closed-form local update $\rightarrow$ low-rank module replacement $\rightarrow$ evaluation.

Two calibration sets are used: 256 samples estimate activation statistics for whitening, while 16 samples provide input-output pairs for error analysis and the closed-form update.

In [ ]:
from dataclasses import dataclass
import copy
import torch
import torch.nn as nn

from utils.model_utils import get_model_from_huggingface, find_layers
from utils.data_utils import get_calib_train_data, get_loaders


@dataclass
class CompressionConfig:
    model_id: str = "/path/to/llama-2-7b-hf"
    dataset: str = "wikitext2"
    overall_compression_ratio: float = 0.20  # R_o: fraction removed
    residual_factor: float = 0.05           # beta
    whitening_nsamples: int = 256
    updating_nsamples: int = 16
    sequence_length: int = 2048
    layer_search_step: int = 2
    seed: int = 3
    device: str = "cuda"


cfg = CompressionConfig()

# Naming convention used throughout this notebook:
#   compression_ratio = fraction of parameters removed
#   keep_ratio        = fraction of parameters retained
# This avoids the repository entry point's implicit ratio inversion.

## 1. Load the model and prepare calibration data

**Purpose.** Load the pretrained causal language model without retraining and construct the two small calibration sets used by subsequent stages.

**Output.** A frozen model, tokenizer, whitening loader, update loader, and the ordered Transformer blocks.

In [ ]:
model, tokenizer = get_model_from_huggingface(cfg.model_id)
model = model.eval().float()
model.config.use_cache = False

# A larger set estimates X^T X for each linear layer.
whitening_loader = get_calib_train_data(
    cfg.dataset,
    tokenizer,
    nsamples=cfg.whitening_nsamples,
    seqlen=cfg.sequence_length,
    seed=cfg.seed,
)

# A smaller set is used for layer-output error and local reconstruction.
update_loader, _ = get_loaders(
    cfg.dataset,
    nsamples=cfg.updating_nsamples,
    seed=cfg.seed,
    seqlen=cfg.sequence_length,
    tokenizer=tokenizer,
)

def transformer_layers(model, model_id):
    if "opt" in model_id.lower():
        return model.model.decoder.layers
    return model.model.layers

layers = transformer_layers(model, cfg.model_id)

## 2. Profile activation-aware scaling matrices

**Motivation.** Weight error alone does not reflect the error observed by the model. For the input activations $X$ of each linear layer, ERC-SVD accumulates

$$G=X^TX,$$

then computes a Cholesky factor $S$ satisfying $G\approx SS^T$. Performing SVD on $WS$ makes the truncation sensitive to the activation distribution.

**Output.** A nested dictionary $\texttt{profiling\_mat[layer\_id][linear\_name]}=S$.

In [ ]:
def stable_cholesky(gram, eps=1e-6):
    """Compute S such that gram is approximately S @ S.T."""
    gram = gram.double()
    try:
        return torch.linalg.cholesky(gram)
    except RuntimeError:
        # Calibration statistics can be singular; shift the spectrum.
        lambda_min = torch.linalg.eigvalsh(gram)[0]
        shift = torch.clamp(-lambda_min + eps, min=eps)
        eye = torch.eye(gram.shape[0], device=gram.device)
        return torch.linalg.cholesky(gram + shift * eye)


def profile_one_layer(layer, layer_inputs, forward_layer):
    """Accumulate X^T X for every Linear module in one block."""
    linears = find_layers(layer)
    grams = {
        name: torch.zeros(module.in_features, module.in_features,
                          device=layer_inputs.device)
        for name, module in linears.items()
    }

    def make_hook(name):
        def hook(_module, inputs, _output):
            X = inputs[0].detach().float().reshape(-1, inputs[0].shape[-1])
            grams[name].add_(X.T @ X)
        return hook

    handles = [module.register_forward_hook(make_hook(name))
               for name, module in linears.items()]
    layer_outputs = forward_layer(layer, layer_inputs)
    for handle in handles:
        handle.remove()

    scaling = {name: stable_cholesky(G).cpu() for name, G in grams.items()}
    return scaling, layer_outputs


# The repository's profile(...) additionally captures first-block inputs,
# attention masks, and position ids, then moves one block at a time to CUDA.
# profiling_mat = profile(cfg.model_id, model, whitening_loader, cfg.device)

## 3. Select how many final layers to compress

**Motivation.** Compressing an early block perturbs the inputs of every later block, so approximation errors accumulate through the network. ERC-SVD keeps the first $N-k$ blocks unchanged and compresses only the final $k$ blocks.

For an overall removed fraction $R_o$, the removed fraction assigned to each selected block is

$$R_l=\frac{N R_o}{k}, \qquad R_l<1.$$

For each candidate $k$, the method measures the relative final-layer error

$$e_N=\frac{\lVert H_N-\widehat H_N\rVert_F}{\lVert H_N\rVert_F},$$

and selects the candidate with the smallest $e_N$.

**Output.** The selected number of compressed layers $k^*$ and their layer-wise compression ratio $R_l^*$.

In [ ]:
def relative_output_error(reference, approximation):
    return (torch.linalg.norm(reference - approximation)
            / torch.linalg.norm(reference)).item()


def select_last_layers(model, overall_ratio, search_step, evaluate_candidate):
    """Algorithm 3: choose k by minimum final-block output error."""
    N = len(transformer_layers(model, cfg.model_id))
    search_log = []

    for k in range(search_step, N, search_step):
        layer_compression_ratio = N * overall_ratio / k
        if layer_compression_ratio >= 1.0:
            continue

        trial_model = copy.deepcopy(model)
        final_error, layer_errors = evaluate_candidate(
            trial_model,
            last_k=k,
            layer_compression_ratio=layer_compression_ratio,
        )
        search_log.append({
            "last_k": k,
            "layer_compression_ratio": layer_compression_ratio,
            "final_error": final_error,
            "layer_errors": layer_errors,
        })

    best = min(search_log, key=lambda item: item["final_error"])
    return best["last_k"], best["layer_compression_ratio"], search_log


# loss_analysis(...) performs one candidate run and records layer-wise errors.
# Repeating step 0 for candidate k values yields the selection above.

## 4. Allocate rank and perform residual-compensated SVD

For a weight $W\in\mathbb{R}^{m\times n}$, define $\alpha=mn/(m+n)$ because a rank-$r$ factorization stores $r(m+n)$ parameters. Given the selected layer compression ratio $R_l$,

$$r=(1-R_l)\alpha,\qquad r_r=\beta\alpha,\qquad r_i=r-r_r.$$

ERC-SVD first constructs $W_i=\operatorname{SVD}_{r_i}(WS)S^{-1}$, then approximates the residual $E=W-W_i$ with rank $r_r$. The final weight is $\widehat W=W_i+E_{r_r}$.

**Output.** Two low-rank factors $A\in\mathbb{R}^{m\times r}$ and $B\in\mathbb{R}^{r\times n}$ such that $\widehat W=AB$ and $r_i+r_r=r$.

In [ ]:
def truncated_svd_factors(M, rank):
    """Return A, B such that A @ B is the rank-r approximation of M."""
    U, sigma, Vt = torch.linalg.svd(M, full_matrices=False)
    sqrt_sigma = torch.diag(torch.sqrt(sigma[:rank]))
    A = U[:, :rank] @ sqrt_sigma
    B = sqrt_sigma @ Vt[:rank, :]
    return A, B


def residual_compensated_svd(W, S, layer_compression_ratio, beta=0.05):
    m, n = W.shape
    alpha = m * n / (m + n)
    target_rank = int((1.0 - layer_compression_ratio) * alpha)
    residual_rank = int(beta * alpha)
    intermediate_rank = target_rank - residual_rank
    if intermediate_rank <= 0:
        raise ValueError("Residual rank exceeds the target rank budget.")

    # Stage 1: activation-aware approximation in the scaled space.
    A_i, B_i_scaled = truncated_svd_factors(W @ S, intermediate_rank)
    B_i = B_i_scaled @ torch.linalg.inv(S)
    W_intermediate = A_i @ B_i

    # Stage 2: spend the reserved rank directly on the residual.
    residual = W - W_intermediate
    A_r, B_r = truncated_svd_factors(residual, residual_rank)

    # Concatenation implements A_i B_i + A_r B_r without dense storage.
    A = torch.cat([A_i, A_r], dim=1)
    B = torch.cat([B_i, B_r], dim=0)

    diagnostics = {
        "target_rank": target_rank,
        "intermediate_rank": intermediate_rank,
        "residual_rank": residual_rank,
        "weight_error": relative_output_error(W, A @ B),
    }
    return A, B, diagnostics


# Addition follows from W = W_intermediate + residual.
# Orthogonality is unnecessary, and rank(A @ B) <= r_i + r_r = r.

## 5. Apply the closed-form local reconstruction update

**Motivation.** Low weight error does not guarantee low layer-output error. After residual compensation, the right factor $B$ is fixed and the left factor $A$ is refitted using calibration input-output pairs.

For flattened inputs $X$ and reference outputs $Y$, solve

$$A^*=\arg\min_A\lVert Y-(XB^T)A^T\rVert_F^2.$$

This least-squares problem requires no gradient-based retraining.

**Output.** Updated factor $A^*$ and reconstruction errors before and after the update.

In [ ]:
def closed_form_left_update(A, B, linear_inputs, reference_outputs):
    """Refit A while keeping the low-rank basis B fixed."""
    X = linear_inputs.reshape(-1, linear_inputs.shape[-1]).float()
    Y = reference_outputs.reshape(-1, reference_outputs.shape[-1]).float()

    latent = X @ B.T
    before = relative_output_error(Y, latent @ A.T)

    # Solve latent @ A_T approximately equal to Y.
    A_T = torch.linalg.lstsq(latent, Y).solution
    A_updated = A_T.T
    after = relative_output_error(Y, latent @ A_updated.T)
    return A_updated, {"before_update": before, "after_update": after}


# local_update uses the equivalent fixed representation X @ V^T @ Sigma
# and refits U in closed form.

## 6. Replace dense projections with low-rank modules

A dense map $y=xW^T+b$ is replaced by $y=(xB^T)A^T+b$. For LLaMA and Mistral, this applies to attention projections $q,k,v,o$ and MLP projections $gate,up,down$. OPT uses $q,k,v,out,fc1,fc2$. Biases and normalization modules are preserved.

**Output.** Transformer blocks whose dense Linear modules are stored and executed as rank-$r$ factor pairs.

In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=False):
        super().__init__()
        self.v_proj = nn.Linear(in_features, rank, bias=False)
        self.u_proj = nn.Linear(rank, out_features, bias=bias)

    def forward(self, x):
        return self.u_proj(self.v_proj(x))


@torch.no_grad()
def install_low_rank_linear(layer, module_name, original, A, B):
    replacement = LowRankLinear(
        original.in_features,
        original.out_features,
        rank=B.shape[0],
        bias=original.bias is not None,
    ).to(device=A.device, dtype=original.weight.dtype)

    replacement.u_proj.weight.copy_(A.to(original.weight.dtype))
    replacement.v_proj.weight.copy_(B.to(original.weight.dtype))
    if original.bias is not None:
        replacement.u_proj.bias.copy_(original.bias)
    layer.set_submodule(module_name, replacement)


# Architecture-specific repository wrappers preserve rotary embeddings,
# masks, KV-cache behavior, biases, and norms while executing v_proj -> u_proj.

## 7. Compress the selected blocks sequentially

The first $N-k$ blocks remain unchanged. For each selected block, ERC-SVD records every projection's input and reference output, computes and locally updates its low-rank factors, replaces the projection, and forwards the compressed block. Its output becomes the next block's input.

This sequential update accounts for error already introduced by earlier compressed blocks.

**Output.** The compressed model plus per-layer and per-projection diagnostics.

In [ ]:
def collect_linear_io(layer, hidden_states, forward_layer):
    cache = {}
    linears = find_layers(layer)

    def make_hook(name):
        def hook(_module, inputs, output):
            cache[name] = (inputs[0].detach(), output.detach())
        return hook

    handles = [module.register_forward_hook(make_hook(name))
               for name, module in linears.items()]
    reference_output = forward_layer(layer, hidden_states)
    for handle in handles:
        handle.remove()
    return cache, reference_output


@torch.no_grad()
def compress_last_k_layers(model, hidden_states, profiling_mat, last_k,
                           layer_compression_ratio, beta, forward_layer):
    layers = transformer_layers(model, cfg.model_id)
    start = len(layers) - last_k
    compression_log = {}

    for layer_id, layer in enumerate(layers):
        if layer_id < start:
            hidden_states = forward_layer(layer, hidden_states)
            continue

        io_cache, _ = collect_linear_io(layer, hidden_states, forward_layer)
        layer_log = {}

        for name, linear in list(find_layers(layer).items()):
            W = linear.weight.detach().float()
            S = profiling_mat[layer_id][name].to(W.device).float()
            A, B, rank_info = residual_compensated_svd(
                W, S, layer_compression_ratio, beta
            )
            X, Y = io_cache[name]
            A, update_info = closed_form_left_update(A, B, X, Y)
            install_low_rank_linear(layer, name, linear, A, B)
            layer_log[name] = {**rank_info, **update_info}

        hidden_states = forward_layer(layer, hidden_states)
        compression_log[layer_id] = layer_log

    return model, compression_log

## 8. End-to-end orchestration, saving, and evaluation

Phase A profiles activations and searches for the best final-layer subset. Phase B reloads a clean model, applies the selected configuration, saves the checkpoint, and evaluates perplexity and zero-shot accuracy. Reloading prevents temporary search models from contaminating the final checkpoint.

In [ ]:
# ------------------------- Phase A: analysis -------------------------
# profiling_mat = profile(cfg.model_id, model, whitening_loader, cfg.device)
# best_k, layer_ratio, search_log = select_last_layers(
#     model, cfg.overall_compression_ratio,
#     cfg.layer_search_step, evaluate_candidate,
# )

# ----------------------- Phase B: compression -----------------------
# model, tokenizer = get_model_from_huggingface(cfg.model_id)
# first_inputs, masks, positions = capture_first_block_inputs(
#     model, update_loader, cfg.device
# )
# compressed_model, compression_log = compress_last_k_layers(
#     model, first_inputs, profiling_mat,
#     last_k=best_k, layer_compression_ratio=layer_ratio,
#     beta=cfg.residual_factor, forward_layer=forward_with_masks,
# )

# checkpoint = {
#     "model": compressed_model,
#     "tokenizer": tokenizer,
#     "selected_last_k": best_k,
#     "layer_compression_ratio": layer_ratio,
#     "search_log": search_log,
#     "compression_log": compression_log,
# }
# torch.save(checkpoint, "erc_svd_compressed_model.pt")

# ---------------------------- Evaluation ----------------------------
# from evaluater import ppl_eval
# ppl_eval(compressed_model, tokenizer,
#          datasets=["wikitext2", "ptb", "c4"],
#          model_seq_len=cfg.sequence_length)
#
# evaluation_zero_acc.py evaluates OpenBookQA, ARC-easy, WinoGrande,
# HellaSwag, ARC-challenge, PIQA, and MathQA via lm-eval-harness.

## Implementation map and interpretation

| Notebook stage | Repository implementation | Main output |
|---|---|---|
| Model and data | utils/model_utils.py, utils/data_utils.py | Model and two calibration loaders |
| Scaling matrices | profile() | Per-layer Cholesky factors $S$ |
| Layer selection | loss_analysis() with candidate layer_num values | Layer-wise and final-layer errors |
| Residual compensation | erc_svd_update | Residual-compensated intermediate weight |
| Closed-form update | local_update | Updated low-rank factors |
| Module replacement | component/svd_llama.py, svd_mistral.py, svd_opt.py | Low-rank Transformer blocks |
| Compression driver | whitening_erc_svd() | Compressed model |
| Evaluation | evaluater.py, evaluation_zero_acc.py | PPL and zero-shot accuracy |

**Implementation note.** The repository encodes several layer-count/ratio combinations as lookup branches and writes error curves to JSON. This notebook presents the intended general algorithm explicitly: enumerate valid $k$, compute $R_l=NR_o/k$, and select the configuration with the lowest final-layer error.

**Scope.** LoRA/PEFT utilities are optional post-compression fine-tuning components, not part of the core post-training ERC-SVD pipeline.